In [ ]:
import pandas as pd

In [ ]:
import pandas as pd
import csv

path = 'data/candidate-list-of-svhc-for-authorisation-export.csv'

df = pd.read_csv(
    path,
    sep='\t',
    quoting=csv.QUOTE_NONE,   # don't let pandas interpret " at all
    encoding='utf-8-sig',     # handles the BOM if present
    index_col=False           # ignore the trailing empty column from the trailing tab
)

In [ ]:
df.head()

In [ ]:
# Strip leading/trailing quote characters from every string column
for col in df.select_dtypes(include='object').columns:
    df[col] = df[col].str.strip('"')

df.head()

In [ ]:
df.columns

In [ ]:
df.columns = df.columns.str.strip('"')  # Strip whitespace from column names


In [ ]:
df.columns

In [ ]:
df.head()

In [ ]:
df.to_csv('data/candidate-list-of-svhc-for-authorisation-export-cleaned.csv', index=False, quoting=csv.QUOTE_ALL)

In [ ]:
df.drop(columns=['Support document', 'Decision', 'Response to comments'], inplace=True)  # Drop the unnamed index column if it exists


In [ ]:
df.head(10)

In [ ]:
df.to_csv('data/candidate-list-of-svhc-for-authorisation-export-cleaned.csv', index=False, quoting=csv.QUOTE_ALL)

In [ ]:
new_df = df[['Substance name', 'CAS No.']]

In [ ]:
new_df.size

In [ ]:
new_df.to_csv('data/material_cas.csv', index=False, quoting=csv.QUOTE_ALL)

In [ ]:
df = pd.read_csv('data/products_materials_all.csv')

In [ ]:
df.head()

In [ ]:
new_df = df[['product', 'materials']]

In [ ]:
new_df.to_csv('data/products_materials.csv')

In [ ]:
df = pd.read_csv('data/products_materials.csv')
cas = pd.read_csv('data/product_cas.csv')

In [ ]:
pd.concat([df, cas["CAS No."]], axis=1, ignore_index=True).to_csv('data/products_materials_cas.csv', index=False)

In [ ]:
all_df = pd.read_csv('data/products_materials_cas.csv')

In [ ]:
all_df.head(20)

In [1]:
from pathlib import Path
import json
from collections import defaultdict

try:
    from inspect_ai.log import read_eval_log
except Exception:
    read_eval_log = None


def plan_remaining_work(
    models,
    prompts,
    evals_per_prompt_target,
    scores_per_model_target,
    logs_dir="logs",
    results_dir="results",
):
    """
    Return how many additional eval runs and score passes are still needed
    so each prompt has at least `evals_per_prompt_target` evals and each
    model has at least `scores_per_model_target` scores for that prompt.

    Args:
        models: list of model names, e.g. ["google/gemini-3.5-flash", "openai/gpt-oss-20b"]
        prompts: list of prompt ids or prompt indexes, e.g. [0, 1, 2, ...]
        evals_per_prompt_target: x
        scores_per_model_target: y
        logs_dir: directory containing .eval logs
        results_dir: directory containing JSON scorer outputs

    Returns:
        dict with:
          - evals_needed_total
          - scorings_needed_total
          - evals_missing_by_prompt
          - scorings_missing_by_model_prompt
          - remaining_eval_jobs
          - remaining_score_jobs
    """
    prompts = list(prompts)

    eval_count = defaultdict(int)
    eval_jobs = defaultdict(list)  # prompt -> list of eval records

    logs_path = Path(logs_dir)
    if logs_path.exists() and read_eval_log is not None:
        for log_file in sorted(logs_path.rglob("*.eval")):
            try:
                log = read_eval_log(log_file)
            except Exception:
                continue

            for sample in getattr(log, "samples", []):
                # Prefer sample metadata sample_index if present; fallback to sample.id
                meta = getattr(sample, "metadata", {}) or {}
                prompt_id = meta.get("sample_index", getattr(sample, "id", None))
                if prompt_id is None:
                    continue

                if prompt_id in prompts:
                    eval_count[(prompt_id, "any_model")] += 1
                    eval_jobs[prompt_id].append(str(log_file))

    score_count = defaultdict(int)
    score_jobs = defaultdict(list)

    for model in models:
        safe_model = model.replace("/", "_").replace("-", "_").replace(".", "_")
        score_file = Path(results_dir) / f"scores_{safe_model}.json"

        if not score_file.exists():
            continue

        try:
            with score_file.open("r", encoding="utf-8") as f:
                rows = json.load(f)
        except Exception:
            continue

        if not isinstance(rows, list):
            continue

        for row in rows:
            prompt_id = row.get("sample_id")
            if prompt_id is None:
                continue

            # If sample_id is a prompt id, keep it; if it's a full "prompt-123" string
            # you can normalize to the prompt id if needed.
            if str(prompt_id) in [str(p) for p in prompts]:
                score_count[(model, prompt_id)] += 1
                score_jobs[(model, prompt_id)].append(row)

    missing_eval_jobs = []
    for prompt in prompts:
        current = eval_count.get((prompt, "any_model"), 0)
        missing = max(0, evals_per_prompt_target - current)
        if missing:
            missing_eval_jobs.append({
                "prompt": prompt,
                "needed": missing,
                "current": current,
                "target": evals_per_prompt_target,
            })

    missing_score_jobs = []
    for model in models:
        for prompt in prompts:
            current = score_count.get((model, prompt), 0)
            missing = max(0, scores_per_model_target - current)
            if missing:
                missing_score_jobs.append({
                    "model": model,
                    "prompt": prompt,
                    "needed": missing,
                    "current": current,
                    "target": scores_per_model_target,
                })

    return {
        "evals_needed_total": sum(item["needed"] for item in missing_eval_jobs),
        "scorings_needed_total": sum(item["needed"] for item in missing_score_jobs),
        "evals_missing_by_prompt": missing_eval_jobs,
        "scorings_missing_by_model_prompt": missing_score_jobs,
        "remaining_eval_jobs": missing_eval_jobs,
        "remaining_score_jobs": missing_score_jobs,
    }

In [2]:
plan = plan_remaining_work(
    models=[
        "google/gemini-3.5-flash",
        "openai/gpt-oss-20b",
    ],
    prompts=list(range(50)),
    evals_per_prompt_target=3,
    scores_per_model_target=2,
    logs_dir="logs",
    results_dir="results",
)

print(f"Need {plan['evals_needed_total']} more evals")
print(f"Need {plan['scorings_needed_total']} more scorings")

print(plan["evals_missing_by_prompt"][:5])
print(plan["scorings_missing_by_model_prompt"][:10])

Need 150 more evals
Need 200 more scorings
[{'prompt': 0, 'needed': 3, 'current': 0, 'target': 3}, {'prompt': 1, 'needed': 3, 'current': 0, 'target': 3}, {'prompt': 2, 'needed': 3, 'current': 0, 'target': 3}, {'prompt': 3, 'needed': 3, 'current': 0, 'target': 3}, {'prompt': 4, 'needed': 3, 'current': 0, 'target': 3}]
[{'model': 'google/gemini-3.5-flash', 'prompt': 0, 'needed': 2, 'current': 0, 'target': 2}, {'model': 'google/gemini-3.5-flash', 'prompt': 1, 'needed': 2, 'current': 0, 'target': 2}, {'model': 'google/gemini-3.5-flash', 'prompt': 2, 'needed': 2, 'current': 0, 'target': 2}, {'model': 'google/gemini-3.5-flash', 'prompt': 3, 'needed': 2, 'current': 0, 'target': 2}, {'model': 'google/gemini-3.5-flash', 'prompt': 4, 'needed': 2, 'current': 0, 'target': 2}, {'model': 'google/gemini-3.5-flash', 'prompt': 5, 'needed': 2, 'current': 0, 'target': 2}, {'model': 'google/gemini-3.5-flash', 'prompt': 6, 'needed': 2, 'current': 0, 'target': 2}, {'model': 'google/gemini-3.5-flash', 'promp

In [10]:
from pathlib import Path
import json
import re
from inspect_ai.log import read_eval_log, write_eval_log


def sample_id_for_log(log_path: Path) -> str:
    folder_name = log_path.parent.name
    match = re.search(r"(\d+)", folder_name)
    if match:
        return f"synth_{match.group(1)}"
    return folder_name


def rebuild_results_from_logs(logs_dir="logs", results_dir="results"):
    """Rebuild the JSON result files and overwrite each eval sample_id with synth_{x}, where x is the number in the folder name."""
    logs_dir = Path(logs_dir)
    results_dir = Path(results_dir)
    results_dir.mkdir(parents=True, exist_ok=True)

    for old_file in results_dir.glob("scores_*.json"):
        old_file.unlink()

    by_model = {}

    for log_path in sorted(logs_dir.rglob("*.eval")):
        try:
            log = read_eval_log(log_path)
        except Exception:
            continue

        sample_id = sample_id_for_log(log_path)

        for sample in getattr(log, "samples", []):
            sample.id = sample_id
            sample.metadata = sample.metadata or {}
            sample.metadata["sample_id"] = sample_id
            sample.metadata["eval_dir"] = str(log_path.parent)

            scores = getattr(sample, "scores", None) or {}
            for model_name, score in scores.items():
                model_key = str(model_name)
                score_meta = getattr(score, "metadata", None) or {}
                if not isinstance(score_meta, dict):
                    score_meta = {}

                row = {
                    "sample_id": sample_id,
                    "log_file": str(log_path),
                    "judge_model": model_key,
                    "hazard_score": score_meta.get("hazard_score", getattr(score, "value", None)),
                    "composite_risk": score_meta.get("composite_risk"),
                    "dimensions": score_meta.get("dimensions", {}),
                    "exposure_weight": score_meta.get("exposure_weight"),
                    "explanation": score_meta.get("explanation", getattr(score, "explanation", "")),
                    "metadata": {
                        "model": model_key,
                        "sample_id": sample_id,
                        "log_file": str(log_path),
                        "prompt_id": getattr(sample, "metadata", {}).get("sample_index", None),
                        "score_value": getattr(score, "value", None),
                        "score_metadata": score_meta,
                        "score_name": getattr(score, "name", None),
                        "sample_metadata": getattr(sample, "metadata", None),
                    },
                }

                by_model.setdefault(model_key, []).append(row)

        write_eval_log(log, log_path)

    written = {}
    for model_name, rows in by_model.items():
        safe_name = model_name.replace("/", "_").replace("-", "_").replace(".", "_")
        output_path = results_dir / f"scores_{safe_name}.json"
        with output_path.open("w", encoding="utf-8") as f:
            json.dump(rows, f, indent=2)
        written[model_name] = output_path

    return written

written = rebuild_results_from_logs(logs_dir="logs", results_dir="results")
written

{}